## import libraries

In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
import os
from pathlib import Path
import re
from shapely.geometry import MultiPolygon

## read datasets

In [2]:
data_root = Path("_data/eurocrops2_original_data")

In [3]:
def get_inventory(root_path):
    inventory = []
    
    for country_dir in root_path.iterdir():
        if country_dir.is_dir():
            for file_path in country_dir.glob("*.parquet"):
                parts = file_path.stem.split('_')
                
                country_id = parts[0]   
                year = int(parts[-1])   
                country = country_dir.name  
                
                inventory.append({"country": country, 
                                  "sub_region": country_id, 
                                  "year": year, 
                                  "id": f"{country}_{year}", 
                                  "sub_region_id": f"{country_id}_{year}", 
                                  "path": file_path})
    
    df = pd.DataFrame(inventory)
    
    if not df.empty:
        df = df.sort_values(by=["country", "year"]).reset_index(drop=True)
        
    return df

In [4]:
available_datasets = get_inventory(data_root)
available_datasets

,country,sub_region,year,id,sub_region_id,path
0,be,be2,2018,be_2018,be2_2018,_data/eurocrops2_original_data/be/be2_2018.par...
1,be,be3,2018,be_2018,be3_2018,_data/eurocrops2_original_data/be/be3_2018.par...
2,be,be2,2019,be_2019,be2_2019,_data/eurocrops2_original_data/be/be2_2019.par...
3,de,de4,2011,de_2011,de4_2011,_data/eurocrops2_original_data/de/de4_2011.par...
4,de,de4,2012,de_2012,de4_2012,_data/eurocrops2_original_data/de/de4_2012.par...
5,de,de4,2019,de_2019,de4_2019,_data/eurocrops2_original_data/de/de4_2019.par...
6,iti,iti1,2019,iti_2019,iti1_2019,_data/eurocrops2_original_data/iti/iti1_2019.p...
7,pt,pt,2019,pt_2019,pt_2019,_data/eurocrops2_original_data/pt/pt_2019.parquet
8,si,si,2021,si_2021,si_2021,_data/eurocrops2_original_data/si/si_2021.parquet
9,sk,sk,2019,sk_2019,sk_2019,_data/eurocrops2_original_data/sk/sk_2019.parquet


## select columns of interest

In [5]:
# see what columns we have on the datasets
gpd.read_parquet(available_datasets['path'][0]).head(3)

,cropfield,original_code,off_id,off_area,area_ha,geometry
0,81008,60,2771541,10047.97,1.004644,"MULTIPOLYGON (((3956128.914 3161252.93, 395613..."
1,84501,60,2786934,100213.83,10.021784,"MULTIPOLYGON (((3820125.362 3124726.983, 38201..."
2,14835,2,2077726,6492.42,0.649302,"MULTIPOLYGON (((3855800.834 3096582.761, 38558..."


In [6]:
columns = ['cropfield', 'original_code', 'geometry']

In [7]:
def load_and_filter_datasets(dataframe, columns_of_interest):
    processed = {}
    
    for _, row in dataframe.iterrows():
        gdf = gpd.read_parquet(row['path'])
        processed[row['sub_region_id']] = gdf[columns_of_interest].copy()
        
    return processed

In [8]:
datasets = load_and_filter_datasets(available_datasets, columns)
datasets.keys()

dict_keys(['be2_2018', 'be3_2018', 'be2_2019', 'de4_2011', 'de4_2012', 'de4_2019', 'iti1_2019', 'pt_2019', 'si_2021', 'sk_2019', 'sk_2020', 'sk_2021'])

## mapping the crops on the dataset

In [9]:
target_cols = ['cropfield', 'original_code', 'geometry', 'hcat4_code', 'hcat4_name']

In [10]:
def load_and_map_datasets(inventory_df, mapping_csv_path, columns_of_interest):
    mapping_df = pd.read_csv(mapping_csv_path)

    mapping_df["original_code"] = mapping_df["original_code"].astype(str).str.strip()
    mapping_df["nuts"] = mapping_df["nuts"].astype(str).str.strip()

    processed = {}

    for _, row in inventory_df.iterrows():
        print(f"Processing {row['sub_region_id']}")
        gdf = gpd.read_parquet(row["path"])

        gdf["original_code"] = gdf["original_code"].astype(str).str.strip()

        country_map = mapping_df[mapping_df["nuts"] == row["sub_region"]]
        if country_map.empty:
            country_map = mapping_df[mapping_df["nuts"] == row["country"]]

        merged_gdf = gdf.merge(country_map[["original_code", 
                                            "hcat4_code", 
                                            "hcat4_name"]], 
                                            on="original_code", 
                                            how="left")
        
        final_cols = [c for c in columns_of_interest if c in merged_gdf.columns]
        processed[row["sub_region_id"]] = merged_gdf[final_cols].copy()
        
    return processed

In [11]:
datasets['de4_2011'].head(3)

,cropfield,original_code,geometry
0,1747,990,"MULTIPOLYGON (((4548107.944 3284424.882, 45481..."
1,97500,452,"MULTIPOLYGON (((4507668.225 3236179.946, 45076..."
2,97712,451,"MULTIPOLYGON (((4532784.539 3253846.936, 45327..."


In [12]:
datasets = load_and_map_datasets(available_datasets, "_data/eurocrops2_original_data/eurocrops.csv", target_cols)
datasets.keys()

Processing be2_2018
Processing be3_2018
Processing be2_2019
Processing de4_2011
Processing de4_2012
Processing de4_2019
Processing iti1_2019
Processing pt_2019
Processing si_2021
Processing sk_2019
Processing sk_2020
Processing sk_2021


dict_keys(['be2_2018', 'be3_2018', 'be2_2019', 'de4_2011', 'de4_2012', 'de4_2019', 'iti1_2019', 'pt_2019', 'si_2021', 'sk_2019', 'sk_2020', 'sk_2021'])

In [13]:
datasets['sk_2019'].head(3)

,cropfield,original_code,geometry,hcat4_code,hcat4_name
0,201548,10069,"MULTIPOLYGON (((5029334.644 2928183.441, 50293...",3310604011,winter_rapeseed_rape
1,33,10080,"MULTIPOLYGON (((4888518.191 2781627.843, 48884...",3310106000,maize_corn_popcorn
2,234779,10222,"MULTIPOLYGON (((4957065.083 2934191.158, 49570...",3390000000,not_known_and_other


## select crops of interest

In [14]:
# # count the most occurent classes
# print(f"Austria most common classes: {austria["EC_hcat_n"].value_counts()[:12]} \n")
# print(f"Sweden most common classes: {sweden["EC_hcat_n"].value_counts()[:12]} \n")
# print(f"Portugal most common classes: {portugal["EC_hcat_n"].value_counts()[:12]} \n")
# print(f"Spain most common classes: {spain["EC_hcat_n"].value_counts()[:20]} \n")
# print(f"Ireland most common classes: {ireland["EC_hcat_n"].value_counts()[:12]} \n")
# print(f"France most common classes: {france["EC_hcat_n"].value_counts()[:12]} \n")

In [15]:
# define crops of interest
crops = ['clover', 'potatoes', 'orchards_fruits', 'rye']

def interest_crop(gdf, crop):
    result = gdf[gdf['hcat4_name'] == crop]
    return result, len(result)

crops_dataset = {}

for dataset_id, gdf in datasets.items():

    for crop in crops:

        var_name = f"{crop}_{dataset_id}"

        subset_gdf, subset_size = interest_crop(gdf, crop)

        if subset_size > 0:
            crops_dataset[var_name] = subset_gdf

In [16]:
print(f"Old dataset keys: {datasets.keys()}")
print(f"New dataset keys: {crops_dataset.keys()}")
print(f"Number of new datasets created: {len(crops_dataset)}")
print(f"New datasets created: {crops_dataset.keys()}")

Old dataset keys: dict_keys(['be2_2018', 'be3_2018', 'be2_2019', 'de4_2011', 'de4_2012', 'de4_2019', 'iti1_2019', 'pt_2019', 'si_2021', 'sk_2019', 'sk_2020', 'sk_2021'])
New dataset keys: dict_keys(['clover_be2_2018', 'potatoes_be2_2018', 'orchards_fruits_be2_2018', 'rye_be2_2018', 'clover_be3_2018', 'potatoes_be3_2018', 'orchards_fruits_be3_2018', 'clover_be2_2019', 'potatoes_be2_2019', 'orchards_fruits_be2_2019', 'rye_be2_2019', 'clover_de4_2011', 'potatoes_de4_2011', 'orchards_fruits_de4_2011', 'clover_de4_2012', 'potatoes_de4_2012', 'orchards_fruits_de4_2012', 'clover_de4_2019', 'potatoes_de4_2019', 'orchards_fruits_de4_2019', 'clover_iti1_2019', 'potatoes_iti1_2019', 'rye_iti1_2019', 'clover_pt_2019', 'potatoes_pt_2019', 'orchards_fruits_pt_2019', 'clover_si_2021', 'potatoes_si_2021', 'orchards_fruits_si_2021', 'clover_sk_2019', 'potatoes_sk_2019', 'orchards_fruits_sk_2019', 'rye_sk_2019', 'clover_sk_2020', 'potatoes_sk_2020', 'orchards_fruits_sk_2020', 'rye_sk_2020', 'clover_sk_2

In [17]:
crops_dataset['potatoes_de4_2011'].head(3)

,cropfield,original_code,geometry,hcat4_code,hcat4_name
986,110805,642,"MULTIPOLYGON (((4451108.948 3341516.804, 44511...",3310300000,potatoes
1034,97094,10006,"MULTIPOLYGON (((4481457.868 3260656.763, 44814...",3310300000,potatoes
1153,114571,642,"MULTIPOLYGON (((4443143.816 3344745.771, 44429...",3310300000,potatoes


## merging back sub regions and add year and country id

In [18]:
target_countries = available_datasets['country'].unique()
target_countries

array(['be', 'de', 'iti', 'pt', 'si', 'sk'], dtype=object)

In [19]:
import pandas as pd
import re

def merge_sub_regions(dataset_dict, countries=['be']):
    if isinstance(countries, str):
        countries = [countries]
        
    merged_results = {}

    for country in countries:
        # Pattern for sub-regions (e.g., _be2_)
        pattern = rf'_{country}\d+_' 
        # Pattern for standard countries (e.g., _pt_)
        standard_pattern = rf'_{country}_'
        
        # 1. Identify sub-region keys
        sub_region_keys = [k for k in dataset_dict.keys() if re.search(pattern, k)]
        
        # 2. Identify standard keys (only if no sub-regions exist for this country)
        if not sub_region_keys:
            # If no be2/be3 found, look for keys like 'clover_pt_2018'
            standard_keys = [k for k in dataset_dict.keys() if re.search(standard_pattern, k)]
            
            for k in standard_keys:
                parts = k.split('_')
                df_temp = dataset_dict[k].copy()
                df_temp['country_id'] = country
                df_temp['year'] = parts[-1]
                merged_results[k] = df_temp
            continue # Move to the next country in the list

        # 3. Process sub-regions (Your original logic)
        groups = {}
        for k in sub_region_keys:
            new_key = re.sub(pattern, f'_{country}_', k)
            
            parts = k.split('_')
            year = parts[-1]        
            country_id = country    
            
            df_temp = dataset_dict[k].copy()
            df_temp['country_id'] = country_id
            df_temp['year'] = year
            
            if new_key not in groups:
                groups[new_key] = []
            groups[new_key].append(df_temp)
            
        for target_name, df_list in groups.items():
            merged_results[target_name] = pd.concat(df_list, ignore_index=True)
            
    return merged_results

In [20]:
# def merge_sub_regions(dataset_dict, countries=['be']):
#     if isinstance(countries, str):
#         countries = [countries]
        
#     merged_results = {}

#     for country in countries:
#         pattern = rf'_{country}\d+_' 
#         keys_for_country = [k for k in dataset_dict.keys() if re.search(pattern, k)]
        
#         groups = {}
#         for k in keys_for_country:
#             new_key = re.sub(pattern, f'_{country}_', k)
            
#             parts = k.split('_')
#             year = parts[-1]        
#             country_id = country    
            
#             df_temp = dataset_dict[k].copy()
#             df_temp['country_id'] = country_id
#             df_temp['year'] = year
            
#             if new_key not in groups:
#                 groups[new_key] = []
#             groups[new_key].append(df_temp)
            
#         for target_name, df_list in groups.items():
#             merged_results[target_name] = pd.concat(df_list, ignore_index=True)
            
#     return merged_results

In [21]:
crops_dataset = merge_sub_regions(crops_dataset, countries=target_countries)
crops_dataset.keys()

dict_keys(['clover_be_2018', 'potatoes_be_2018', 'orchards_fruits_be_2018', 'rye_be_2018', 'clover_be_2019', 'potatoes_be_2019', 'orchards_fruits_be_2019', 'rye_be_2019', 'clover_de_2011', 'potatoes_de_2011', 'orchards_fruits_de_2011', 'clover_de_2012', 'potatoes_de_2012', 'orchards_fruits_de_2012', 'clover_de_2019', 'potatoes_de_2019', 'orchards_fruits_de_2019', 'clover_iti_2019', 'potatoes_iti_2019', 'rye_iti_2019', 'clover_pt_2019', 'potatoes_pt_2019', 'orchards_fruits_pt_2019', 'clover_si_2021', 'potatoes_si_2021', 'orchards_fruits_si_2021', 'clover_sk_2019', 'potatoes_sk_2019', 'orchards_fruits_sk_2019', 'rye_sk_2019', 'clover_sk_2020', 'potatoes_sk_2020', 'orchards_fruits_sk_2020', 'rye_sk_2020', 'clover_sk_2021', 'potatoes_sk_2021', 'orchards_fruits_sk_2021', 'rye_sk_2021'])

In [22]:
crops_dataset['orchards_fruits_sk_2020'].head(3)

,cropfield,original_code,geometry,hcat4_code,hcat4_name,country_id,year
1263,1150,10126,"MULTIPOLYGON (((4911637.867 2820433.614, 49116...",3330100000,orchards_fruits,sk,2020
3679,144556,10126,"MULTIPOLYGON (((4881535.825 2824009.538, 48815...",3330100000,orchards_fruits,sk,2020
3866,198073,10126,"MULTIPOLYGON (((4895722.068 2888390.336, 48957...",3330100000,orchards_fruits,sk,2020


## eliminate multipoligons with more than one poligon

In [23]:
def drop_multipoligons(datasets):
    cleaned_results = {}
    
    for name, gdf in datasets.items():
        multi_parts_indices = []
        eliminates = 0
        
        for idx, geom in gdf['geometry'].items():
            if isinstance(geom, MultiPolygon) and len(geom.geoms) > 1:
                multi_parts_indices.append(idx)
                eliminates = eliminates + 1
        
        print(f"Eliminating multipoligon {eliminates} poligons in '{name}'.")
        
        if multi_parts_indices:
            cleaned_gdf = gdf.drop(index=multi_parts_indices)
            cleaned_results[name] = cleaned_gdf.reset_index(drop=True)
        else:
            cleaned_results[name] = gdf.copy()
            
    return cleaned_results

In [24]:
crops_dataset = drop_multipoligons(crops_dataset)

Eliminating multipoligon 16 poligons in 'clover_be_2018'.
Eliminating multipoligon 8 poligons in 'potatoes_be_2018'.
Eliminating multipoligon 0 poligons in 'orchards_fruits_be_2018'.
Eliminating multipoligon 0 poligons in 'rye_be_2018'.
Eliminating multipoligon 19 poligons in 'clover_be_2019'.
Eliminating multipoligon 16 poligons in 'potatoes_be_2019'.
Eliminating multipoligon 0 poligons in 'orchards_fruits_be_2019'.
Eliminating multipoligon 0 poligons in 'rye_be_2019'.
Eliminating multipoligon 75 poligons in 'clover_de_2011'.
Eliminating multipoligon 2 poligons in 'potatoes_de_2011'.
Eliminating multipoligon 33 poligons in 'orchards_fruits_de_2011'.
Eliminating multipoligon 96 poligons in 'clover_de_2012'.
Eliminating multipoligon 29 poligons in 'potatoes_de_2012'.
Eliminating multipoligon 12 poligons in 'orchards_fruits_de_2012'.
Eliminating multipoligon 17 poligons in 'clover_de_2019'.
Eliminating multipoligon 5 poligons in 'potatoes_de_2019'.
Eliminating multipoligon 3 poligons in 

## get centroid and long and lat

In [25]:
def access_coords(gpd, index):
    poligon = gpd.geometry.iloc[index]

    coords = list(poligon.exterior.coords)

    coords_numpy = np.array(poligon.exterior.coords)
    return poligon, coords, coords_numpy

In [26]:
def get_centroid(gdf, centroid_as_main=False):
    gdf = gdf.copy()
    local_utm = gdf.estimate_utm_crs()
    gdf['centroid'] = gdf.geometry.to_crs(local_utm).centroid.to_crs(epsg=4326)
    gdf = gdf.to_crs(epsg=4326)
    
    if centroid_as_main: 
        gdf = gdf.set_geometry('centroid')
        
    return gdf

In [27]:
def get_long_lat(gdf):
    gdf['long'] = gdf['centroid'].x
    gdf['lat'] = gdf['centroid'].y
    return gdf

In [28]:
crops_dataset['clover_pt_2019'].head(3)

,cropfield,original_code,geometry,hcat4_code,hcat4_name,country_id,year
23854,93239,46,"MULTIPOLYGON (((2737774.497 1935312.748, 27377...",3310202030,clover,pt,2019
23855,139915,46,"MULTIPOLYGON (((2737779.072 1935303.942, 27378...",3310202030,clover,pt,2019
24346,139916,46,"MULTIPOLYGON (((2737984.421 1935747.743, 27379...",3310202030,clover,pt,2019


In [29]:
# check before applying transformations
print(crops_dataset['clover_pt_2019'].crs)
crops_dataset['clover_pt_2019'].head(3)

{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "ETRS89-extended / LAEA Europe", "base_crs": {"name": "ETRS89", "datum_ensemble": {"name": "European Terrestrial Reference System 1989 ensemble", "members": [{"name": "European Terrestrial Reference Frame 1989"}, {"name": "European Terrestrial Reference Frame 1990"}, {"name": "European Terrestrial Reference Frame 1991"}, {"name": "European Terrestrial Reference Frame 1992"}, {"name": "European Terrestrial Reference Frame 1993"}, {"name": "European Terrestrial Reference Frame 1994"}, {"name": "European Terrestrial Reference Frame 1996"}, {"name": "European Terrestrial Reference Frame 1997"}, {"name": "European Terrestrial Reference Frame 2000"}, {"name": "European Terrestrial Reference Frame 2005"}, {"name": "European Terrestrial Reference Frame 2014"}], "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}, "accuracy": "0.1", "id": {"authority":

,cropfield,original_code,geometry,hcat4_code,hcat4_name,country_id,year
23854,93239,46,"MULTIPOLYGON (((2737774.497 1935312.748, 27377...",3310202030,clover,pt,2019
23855,139915,46,"MULTIPOLYGON (((2737779.072 1935303.942, 27378...",3310202030,clover,pt,2019
24346,139916,46,"MULTIPOLYGON (((2737984.421 1935747.743, 27379...",3310202030,clover,pt,2019


In [30]:
for i in crops_dataset:
    print(f"Apply functions on {i}")
    crops_dataset[i] = get_centroid(crops_dataset[i])
    crops_dataset[i] = get_long_lat(crops_dataset[i])

Apply functions on clover_be_2018
Apply functions on potatoes_be_2018
Apply functions on orchards_fruits_be_2018
Apply functions on rye_be_2018
Apply functions on clover_be_2019
Apply functions on potatoes_be_2019
Apply functions on orchards_fruits_be_2019
Apply functions on rye_be_2019
Apply functions on clover_de_2011
Apply functions on potatoes_de_2011
Apply functions on orchards_fruits_de_2011
Apply functions on clover_de_2012
Apply functions on potatoes_de_2012
Apply functions on orchards_fruits_de_2012
Apply functions on clover_de_2019
Apply functions on potatoes_de_2019
Apply functions on orchards_fruits_de_2019
Apply functions on clover_iti_2019
Apply functions on potatoes_iti_2019
Apply functions on rye_iti_2019
Apply functions on clover_pt_2019
Apply functions on potatoes_pt_2019
Apply functions on orchards_fruits_pt_2019
Apply functions on clover_si_2021
Apply functions on potatoes_si_2021
Apply functions on orchards_fruits_si_2021
Apply functions on clover_sk_2019
Apply fun

In [31]:
# check after applying transformations
print(crops_dataset['clover_pt_2019'].crs)
crops_dataset['clover_pt_2019'].head(3)

EPSG:4326


,cropfield,original_code,geometry,hcat4_code,hcat4_name,country_id,year,centroid,long,lat
23854,93239,46,"MULTIPOLYGON (((-8.3016 38.78175, -8.30159 38....",3310202030,clover,pt,2019,POINT (-8.30185 38.77979),-8.301848,38.779790
23855,139915,46,"MULTIPOLYGON (((-8.30152 38.78168, -8.3013 38....",3310202030,clover,pt,2019,POINT (-8.2993 38.77983),-8.299301,38.779826
24346,139916,46,"MULTIPOLYGON (((-8.30032 38.78602, -8.30017 38...",3310202030,clover,pt,2019,POINT (-8.29858 38.78396),-8.298583,38.783957


## export data

In [32]:
export_folder = "_data/crop_country_exports"

if not os.path.exists(export_folder):
    os.makedirs(export_folder)
    print(f"Created directory: {export_folder}")

for name in crops_dataset.keys():
    temp_df = pd.DataFrame(crops_dataset[name][['country_id', 'year', 'hcat4_code', 
                                                'hcat4_name', 'centroid', 'long', 'lat']]).copy()
    
    file_path = f"{export_folder}/{name}.csv"
    
    temp_df.to_csv(file_path, index=False)
    print(f"Exported: {file_path}")

Created directory: _data/crop_country_exports
Exported: _data/crop_country_exports/clover_be_2018.csv
Exported: _data/crop_country_exports/potatoes_be_2018.csv
Exported: _data/crop_country_exports/orchards_fruits_be_2018.csv
Exported: _data/crop_country_exports/rye_be_2018.csv
Exported: _data/crop_country_exports/clover_be_2019.csv
Exported: _data/crop_country_exports/potatoes_be_2019.csv
Exported: _data/crop_country_exports/orchards_fruits_be_2019.csv
Exported: _data/crop_country_exports/rye_be_2019.csv
Exported: _data/crop_country_exports/clover_de_2011.csv
Exported: _data/crop_country_exports/potatoes_de_2011.csv
Exported: _data/crop_country_exports/orchards_fruits_de_2011.csv
Exported: _data/crop_country_exports/clover_de_2012.csv
Exported: _data/crop_country_exports/potatoes_de_2012.csv
Exported: _data/crop_country_exports/orchards_fruits_de_2012.csv
Exported: _data/crop_country_exports/clover_de_2019.csv
Exported: _data/crop_country_exports/potatoes_de_2019.csv
Exported: _data/cr